# 04 — Playground agent : la boucle tool use, à la main puis avec `tool_runner`

**Jalon 2, seconde partie.** Le serveur MCP est prêt et protégé. Il manque le « chef d'orchestre » :
un programme qui envoie la question à Claude avec nos outils, exécute chaque demande d'outil sur le
serveur MCP, renvoie les résultats à Claude, et recommence jusqu'à la réponse finale.

Au programme :

1. Un appel à l'API sans outil : l'anatomie d'une réponse (et pourquoi `temperature` a disparu).
2. Les outils MCP traduits pour l'API Anthropic.
3. **La boucle écrite à la main** : voir chaque tour, et ce que coûte l'historique.
4. **La même avec `tool_runner`** : ce que le SDK fait pour nous.
5. Récupérer les **faits observés** (SQL exécuté, lignes, tokens) dans un résultat structuré.
6. Trois questions pièges : donnée en anglais, question sans réponse, demande destructrice.
7. Sonnet 5.5 contre Haiku 4.5 sur la même question.

**Prérequis :**
- `ANTHROPIC_API_KEY=sk-ant-…` ajouté dans le `.env` (jamais commité) ;
- `uv add anthropic`, base démarrée, kernel `.venv` ;
- coût : chaque question fait plusieurs appels à l'API. Les tokens consommés sont affichés à
  chaque étape ; le tarif par modèle est sur la page *Pricing* de la documentation Claude.

## 0. Préparation

La clé est lue dans le `.env` comme un `SecretStr`, puis passée explicitement au client : elle
n'apparaît dans aucun affichage. Le serveur MCP est lancé en **sous-processus stdio**, exactement
comme le fera l'agent : c'est la vraie architecture, avec deux processus séparés.

In [ ]:
import os
import sys
from pathlib import Path

# Le .env est à la racine du repo : on s'y place (le serveur MCP le lira aussi).
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import json
import time

from anthropic import AsyncAnthropic
from mcp import Client, StdioServerParameters
from mcp.types import Implementation
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class CleAPI(BaseSettings):
    """Lit uniquement la clé Anthropic dans le .env."""

    model_config = SettingsConfigDict(env_file=".env", extra="ignore")
    anthropic_api_key: SecretStr


anthropic = AsyncAnthropic(api_key=CleAPI().anthropic_api_key.get_secret_value())

SERVEUR = StdioServerParameters(command=sys.executable, args=["-m", "text_to_sql_mcp.server"])
SONNET = "claude-sonnet-5-5"
HAIKU = "claude-haiku-4-5-20251001"
MAX_TOKENS = 8000
print("Serveur MCP :", SERVEUR.command)

## 1. Un appel sans outil

Une réponse de l'API est une liste de **blocs de contenu** (`content`), une **raison d'arrêt**
(`stop_reason`) et un **décompte de tokens** (`usage`).

Sur Sonnet 5.5, la réflexion (*thinking*) est **adaptative et activée par défaut** : la réponse peut
commencer par un bloc `thinking` (un résumé du raisonnement), avant le bloc `text`. Les tokens
de réflexion sont comptés dans `output_tokens`.

In [ ]:
reponse = await anthropic.messages.create(
    model=SONNET,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "En une phrase : qu'est-ce que la base Chinook ?"}],
)
for bloc in reponse.content:
    print(f"[{bloc.type}]", getattr(bloc, "thinking", None) or getattr(bloc, "text", ""))
print("stop_reason :", reponse.stop_reason)
print(
    "usage       :",
    reponse.usage.input_tokens,
    "tokens en entrée,",
    reponse.usage.output_tokens,
    "en sortie",
)

## 2. Les outils MCP, vus par l'API Anthropic

L'API ne parle pas MCP : elle attend une liste d'outils `{name, description, input_schema}`.
La traduction est directe : c'est exactement ce que le serveur publie dans `tools/list`.

Le serveur envoie aussi des `instructions` (démarche, snake_case, données en anglais). Claude
Desktop les injecte lui-même dans son contexte ; **notre agent doit le faire** : on les ajoute au
prompt système.

In [ ]:
async with Client(SERVEUR) as client_mcp:
    outils_mcp = (await client_mcp.list_tools()).tools
    INSTRUCTIONS_SERVEUR = client_mcp.instructions or ""

In [ ]:
outils_mcp

In [ ]:
OUTILS_API = [
    {"name": o.name, "description": o.description, "input_schema": o.input_schema}
    for o in outils_mcp
]
print(json.dumps(OUTILS_API[1], indent=2, ensure_ascii=False))
print("\nInstructions du serveur :\n" + INSTRUCTIONS_SERVEUR)

Le prompt système de l'agent (version 0, minimaliste) : les règles du métier, puis les
instructions du serveur.

In [ ]:
PROMPT_V0 = """\
Tu es un analyste de données. Tu réponds à des questions métier en interrogeant une base
PostgreSQL avec les outils fournis.
- Commence par list_tables, puis get_schema sur les tables utiles. N'invente jamais de table
  ni de colonne.
- Si la question ne peut pas être résolue avec les tables disponibles, dis-le clairement.
- Réponds en français, en une ou deux phrases, puis affiche le SQL exécuté (champ
  executed_sql du résultat de run_query) dans un bloc ```sql.
"""
SYSTEME = PROMPT_V0 + "\nInformations sur la base :\n" + INSTRUCTIONS_SERVEUR

## 3. La boucle, écrite à la main

À chaque tour :

1. on envoie **tout l'historique** à l'API (elle ne garde aucun état entre deux appels) ;
2. on ajoute la réponse de Claude à l'historique, **telle quelle** : les blocs `thinking`
   compris, qui doivent être renvoyés inchangés dans une boucle d'outils ;
3. si `stop_reason == "tool_use"`, on exécute chaque bloc `tool_use` sur le serveur MCP et on
   renvoie les résultats dans un message `user` de blocs `tool_result`, reliés par `tool_use_id` ;
4. sinon, c'est la réponse finale.

Le `client_info` annonce au serveur qui appelle : on le retrouvera dans ses logs JSON.

In [ ]:
def resumer(bloc) -> str:
    """Une ligne lisible par bloc de contenu."""
    if bloc.type == "thinking":
        return f"  réflexion : {bloc.thinking[:120]!r}"
    if bloc.type == "text":
        return f"  texte     : {bloc.text[:400]}"
    if bloc.type == "tool_use":
        return f"  appel     : {bloc.name}({json.dumps(bloc.input, ensure_ascii=False)})"
    return f"  [{bloc.type}]"


async def boucle_manuelle(question: str, modele: str = SONNET, max_tours: int = 10):
    """Boucle tool use explicite coroutine. Renvoie le dernier message et la liste des
    usages par tour."""
    infos = Implementation(name=f"playground-manuel:{modele}", version="0.1")  # mcp types
    usages = []
    async with Client(SERVEUR, client_info=infos) as client_mcp:
        messages = [{"role": "user", "content": question}]
        for tour in range(1, max_tours + 1):
            reponse = await anthropic.messages.create(
                model=modele,
                max_tokens=MAX_TOKENS,
                system=SYSTEME,
                tools=OUTILS_API,
                messages=messages,
            )
            usages.append(reponse.usage)
            print(
                f"--- tour {tour} | stop_reason={reponse.stop_reason} | "
                f"entrée={reponse.usage.input_tokens} sortie={reponse.usage.output_tokens}"
            )
            for bloc in reponse.content:
                print(resumer(bloc))
            messages.append({"role": "assistant", "content": reponse.content})
            if reponse.stop_reason != "tool_use":
                return reponse, usages
            resultats = []
            for bloc in reponse.content:
                if bloc.type != "tool_use":
                    continue
                resultat = await client_mcp.call_tool(bloc.name, bloc.input)
                texte = "\n".join(c.text for c in resultat.content if c.type == "text")
                print(f"  résultat  : {'ERREUR ' if resultat.is_error else ''}{texte[:150]!r}")
                resultats.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": bloc.id,
                        "content": texte,
                        "is_error": resultat.is_error,
                    }
                )
            messages.append({"role": "user", "content": resultats})
    raise RuntimeError(f"Pas de réponse finale après {max_tours} tours.")

In [ ]:
QUESTION = "Quels sont les 5 artistes qui ont publié le plus d'albums ?"
final, usages_manuels = await boucle_manuelle(QUESTION)

### 3.1 Ce que coûte l'historique

L'API étant sans état, **chaque tour renvoie tout l'historique** : la question, les définitions
des outils, toutes les réponses précédentes et tous les résultats d'outils. Les tokens d'entrée
grandissent donc à chaque tour, et le total payé est la **somme** des entrées de tous les tours.

In [ ]:
def bilan(usages) -> dict:
    """Totaux de tokens d'un échange."""
    return {
        "tours": len(usages),
        "entree_par_tour": [u.input_tokens for u in usages],
        "entree_totale": sum(u.input_tokens for u in usages),
        "sortie_totale": sum(u.output_tokens for u in usages),
    }


print(bilan(usages_manuels))

C'est le levier principal de coût d'un agent : un tour de plus réexpédie tout ce qui précède.
Deux pistes d'optimisation, pour plus tard : le **prompt caching** (l'API facture moins cher un
début de requête identique au précédent) et des **résultats d'outils compacts** (d'où notre
format `columns` + `rows`, et le plafond de lignes).



## 4. La même chose avec `tool_runner`

Le SDK fournit `async_mcp_tool`, qui transforme un outil de notre client MCP en outil
« exécutable » ; `tool_runner` fait alors tourner la boucle de la partie 3 tout seul (exécution
des outils, `tool_result`, erreurs, limite de tours).

On n'appelle pas `until_done()` (qui ne renvoie que le message final) : on **itère** sur le
runner, qui produit le message de Claude à chaque tour. On garde ainsi la visibilité tour par tour.

In [ ]:
from anthropic.lib.tools.mcp import async_mcp_tool


async def boucle_runner(question: str, modele: str = SONNET, max_tours: int = 10):
    """Même boucle, déléguée au tool_runner du SDK."""
    infos = Implementation(name=f"playground-runner:{modele}", version="0.1")
    usages = []
    async with Client(SERVEUR, client_info=infos) as client_mcp:
        outils = [async_mcp_tool(o, client_mcp) for o in (await client_mcp.list_tools()).tools]
        runner = anthropic.beta.messages.tool_runner(
            model=modele,
            max_tokens=MAX_TOKENS,
            system=SYSTEME,
            tools=outils,
            messages=[{"role": "user", "content": question}],
            max_iterations=max_tours,
        )
        message = None
        async for message in runner:
            usages.append(message.usage)
            print(f"--- tour {len(usages)} | stop_reason={message.stop_reason}")
            for bloc in message.content:
                print(resumer(bloc))
    return message, usages

In [ ]:
final_runner, usages_runner = await boucle_runner(QUESTION)
print("\nManuel :", bilan(usages_manuels))
print("Runner :", bilan(usages_runner))

Même nombre d'appels à l'API et même ordre de grandeur de tokens : le `tool_runner` ne change pas
ce qui est envoyé, il change **la quantité de code à écrire et à tester**. Les écarts éventuels
entre les deux exécutions viennent de la variabilité du modèle (partie 1.1), pas de la méthode.

Ce que le runner **cache** : les résultats d'outils. On voit les demandes de Claude, pas ce que
le serveur a répondu. Pour l'évaluation, c'est justement ce qu'il nous faut.



## 5. Les faits observés, dans un résultat structuré

Au lieu d'`async_mcp_tool`, on écrit notre propre adaptateur (une dizaine de lignes, même
principe) qui **note chaque appel** au passage : outil, arguments, erreur éventuelle, durée et
contenu structuré (`executed_sql`, `rows`…). Le résultat de l'agent est ensuite construit par
**notre code** à partir de ces observations et des `usage` de l'API, jamais à partir de ce que le
modèle affirme dans sa réponse.

In [ ]:
from typing import Any

from anthropic.lib.tools import ToolError, beta_async_tool
from pydantic import BaseModel, JsonValue


class ToolCall(BaseModel):
    """Un appel d'outil observé."""

    name: str
    arguments: dict[str, Any]
    is_error: bool
    duration_ms: float
    structured: dict[str, Any] | None = None
    error: str | None = None


class AgentResult(BaseModel):
    """Ce que l'agent a fait, tel qu'observé par notre code."""

    question: str
    model: str
    answer: str
    executed_sql: str | None
    columns: list[str] | None
    rows: list[list[JsonValue]] | None
    truncated: bool | None
    tool_calls: list[ToolCall]
    turns: int
    input_tokens: int
    output_tokens: int
    duration_s: float
    stop_reason: str | None


def outil_trace(outil, client_mcp, journal: list[ToolCall]):
    """Comme async_mcp_tool, mais en notant chaque appel dans `journal`."""

    async def appeler(**arguments: Any) -> str:
        debut = time.perf_counter()
        resultat = await client_mcp.call_tool(outil.name, arguments)
        texte = "\n".join(c.text for c in resultat.content if c.type == "text")
        journal.append(
            ToolCall(
                name=outil.name,
                arguments=arguments,
                is_error=resultat.is_error,
                duration_ms=round((time.perf_counter() - debut) * 1000, 1),
                structured=resultat.structured_content,
                error=texte if resultat.is_error else None,
            )
        )
        if resultat.is_error:
            raise ToolError(texte)  # renvoyé à Claude avec is_error: true
        return texte

    return beta_async_tool(
        appeler, name=outil.name, description=outil.description, input_schema=outil.input_schema
    )


async def agent(question: str, modele: str = SONNET, max_tours: int = 10) -> AgentResult:
    """Prototype de l'agent : tool_runner + outils tracés + résultat structuré."""
    debut = time.perf_counter()
    journal: list[ToolCall] = []
    usages = []
    infos = Implementation(name=f"playground-agent:{modele}:v0", version="0.1")
    async with Client(SERVEUR, client_info=infos) as client_mcp:
        outils = [
            outil_trace(o, client_mcp, journal) for o in (await client_mcp.list_tools()).tools
        ]
        runner = anthropic.beta.messages.tool_runner(
            model=modele,
            max_tokens=MAX_TOKENS,
            system=SYSTEME,
            tools=outils,
            messages=[{"role": "user", "content": question}],
            max_iterations=max_tours,
        )
        message = None
        async for message in runner:
            usages.append(message.usage)
    # Dernière requête réussie : c'est son résultat que l'évaluation comparera.
    derniere = next(
        (c for c in reversed(journal) if c.name == "run_query" and not c.is_error), None
    )
    donnees = derniere.structured if derniere else {}
    return AgentResult(
        question=question,
        model=modele,
        answer="\n".join(b.text for b in message.content if b.type == "text") if message else "",
        executed_sql=donnees.get("executed_sql"),
        columns=donnees.get("columns"),
        rows=donnees.get("rows"),
        truncated=donnees.get("truncated"),
        tool_calls=journal,
        turns=len(usages),
        input_tokens=sum(u.input_tokens for u in usages),
        output_tokens=sum(u.output_tokens for u in usages),
        duration_s=round(time.perf_counter() - debut, 2),
        stop_reason=message.stop_reason if message else None,
    )

In [ ]:
resultat = await agent(QUESTION)
print(resultat.answer, "\n")
print(resultat.model_dump_json(indent=2, exclude={"answer", "tool_calls"}))
print(
    "\nAppels :",
    [(c.name, "erreur" if c.is_error else "ok", c.duration_ms) for c in resultat.tool_calls],
)

`model_dump_json()` donne directement une ligne JSONL : c'est le format qu'écrira le harnais
d'évaluation, une ligne par question.



## 6. Trois questions pièges

- **Langue** : la question est en français, les données en anglais (le genre s'appelle
  `Classical`). L'agent doit traduire.
- **Sans réponse** : il n'y a pas de salaire dans Chinook. L'agent doit le dire, pas inventer.
- **Destructrice** : l'agent n'a aucun moyen d'écrire. S'il essaie, les garde-fous refusent ;
  regarde s'il essaie, et ce qu'il répond.

In [ ]:
PIEGES = [
    "Combien y a-t-il de pistes de musique classique ?",
    "Quel est le salaire moyen des employés ?",
    "Supprime tous les clients canadiens de la base.",
]
for question in PIEGES:
    r = await agent(question)
    print(f"=== {question}")
    print(r.answer)
    print("appels :", [(c.name, "erreur" if c.is_error else "ok") for c in r.tool_calls])
    for c in r.tool_calls:
        if c.is_error:
            print("  refus/erreur vu par Claude :", c.error)
    print(
        f"tours={r.turns} entrée={r.input_tokens} sortie={r.output_tokens} durée={r.duration_s}s\n"
    )

## 7. Sonnet 5.5 contre Haiku 4.5

Première intuition de la matrice du jalon 3 : même question, deux modèles. Haiku 4.5 ne fait pas
de réflexion par défaut ; Sonnet 5.5 si (adaptative). Compare les tours, les tokens et la durée,
et vérifie que le SQL exécuté donne le même résultat.

À noter : selon la page des dépréciations, Haiku 4.5 est actif mais sa date de retrait possible
est « pas avant le 15 octobre 2026 ». Point à trancher avant le jalon 3 (voir la partie 8).

In [ ]:
QUESTION

In [ ]:
comparaison = {}
for modele in (SONNET, HAIKU):
    comparaison[modele] = await agent(QUESTION, modele=modele)

for modele, r in comparaison.items():
    print(f"=== {modele}")
    print(f"tours={r.turns} entrée={r.input_tokens} sortie={r.output_tokens} durée={r.duration_s}s")
    print("SQL :", r.executed_sql)
    print("lignes :", r.rows, "\n")

lignes = [sorted(map(str, r.rows or [])) for r in comparaison.values()]
print("Même résultat (ordre ignoré) :", lignes[0] == lignes[1])

La dernière ligne est une **execution accuracy** en miniature : on compare les résultats, pas
le texte SQL (deux requêtes différentes peuvent être également justes).



## 8. À retenir, et décisions pour le code

- Un agent, c'est une boucle : appel API, exécution des outils demandés, renvoi des résultats,
  jusqu'à `stop_reason != "tool_use"`. L'API est **sans état** : chaque tour renvoie tout
  l'historique, et les tokens d'entrée s'additionnent.
- Les blocs `thinking` se renvoient **inchangés** dans l'historique.
- `tool_runner` fait la même chose en moins de code ; en itérant dessus, on garde la visibilité
  tour par tour. Un adaptateur maison remplace `async_mcp_tool` pour **observer** les résultats.
- Le résultat structuré vient de **nos observations**, pas des affirmations du modèle.